In [1]:

import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    silhouette_score
)


In [2]:

# Load datasets
train = pd.read_csv("../training_image_features_grouped.csv")
test = pd.read_csv("../testing_image_features_grouped.csv")

TARGET = "Condition_Class"
CLASSES = ["Excellent", "Good", "Fair", "Poor"]

print("Training Dataset Shape:", train.shape)
print("Testing Dataset Shape:", test.shape)

print("\nTraining Dataset:")
display(train.head())


Training Dataset Shape: (11465, 43)
Testing Dataset Shape: (2644, 43)

Training Dataset:


,Sample_ID,Image_ID,Material_Type,Material_Subtype,Source,Exposure_Type,Crack_Level,Surface_Damage,Breakage_Level,Discoloration_Level,...,dark_pixel_ratio,bright_pixel_ratio,mean_hue,mean_saturation,saturation_std,contour_area,contour_perimeter,circularity,solidity,extent
0,TRAINING_00013,10.jpg,brick,Standard,Demolition_Site,Exterior_Weathered,5,3.5,2,Moderate,...,0.013321,0.141387,16.121604,106.880770,22.460271,16131.0,745.470123,0.364763,0.928938,0.768143
1,TRAINING_00014,10.jpg,concrete,Reinforced,Demolition_Site,Exterior_Weathered,5,4.0,3,High,...,0.017225,0.326960,19.273151,94.502765,23.173802,6688.0,441.102593,0.431943,0.901408,0.557891
2,TRAINING_00015,10.jpg,tile,Standard,Demolition_Site,Exterior_Weathered,3,3.5,4,High,...,0.118015,0.170967,29.613107,51.448496,24.297176,10975.5,720.884337,0.265401,0.810628,0.683917
3,TRAINING_00016,10.jpg,brick,Standard,Demolition_Site,Exterior_Weathered,5,3.5,3,Moderate,...,0.009063,0.008142,13.644708,119.697837,26.109004,19055.5,769.251868,0.404663,0.893785,0.598101
4,TRAINING_00017,10.jpg,concrete,Reinforced,Demolition_Site,Exterior_Weathered,5,3.5,3,Moderate,...,0.000417,0.306625,19.245853,95.316954,25.873127,7753.5,720.364566,0.187760,0.805977,0.488502


In [3]:

# Exclude target-related scores
exclude = {
    "Condition_Score",
    "Damage_Score",
    "Alternative_Use_Suitability"
}

# Select numeric image features
from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

print("Number of Original Features:", len(features))
print("\nSelected Features:")
print(features)


Number of Original Features: 26

Selected Features:
['Crack_Level', 'Surface_Damage', 'Breakage_Level', 'Deformation_Level', 'pixel_img_width', 'pixel_img_height', 'pixel_obj_width', 'pixel_obj_height', 'relative_area', 'edge_density', 'fill_ratio', 'color_std', 'mean_brightness', 'std_brightness', 'contrast', 'texture_laplacian_var', 'dark_pixel_ratio', 'bright_pixel_ratio', 'mean_hue', 'mean_saturation', 'saturation_std', 'contour_area', 'contour_perimeter', 'circularity', 'solidity', 'extent']


In [4]:

# Extract selected features
X_train = train[features].replace([np.inf, -np.inf], np.nan)
X_test = test[features].replace([np.inf, -np.inf], np.nan)

# Handle missing values
imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Standardize features
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Original Training Data Shape:", X_train.shape)
print("Original Testing Data Shape:", X_test.shape)
print("Preprocessing Completed Successfully")


Original Training Data Shape: (11465, 26)
Original Testing Data Shape: (2644, 26)
Preprocessing Completed Successfully


In [5]:

# Initialize PCA
pca = PCA(
    n_components=0.95,
    svd_solver="full"
)

# Fit PCA on training data
X_train_pca = pca.fit_transform(X_train)

# Transform testing data
X_test_pca = pca.transform(X_test)

print("PCA Transformation Completed")

print("\nOriginal Number of Features:", X_train.shape[1])

print("Reduced Number of Features:", X_train_pca.shape[1])

print("\nTraining PCA Shape:", X_train_pca.shape)
print("Testing PCA Shape:", X_test_pca.shape)

print(
    "\nTotal Explained Variance:",
    pca.explained_variance_ratio_.sum()
)


PCA Transformation Completed

Original Number of Features: 26
Reduced Number of Features: 13

Training PCA Shape: (11465, 13)
Testing PCA Shape: (2644, 13)

Total Explained Variance: 0.9534443186321065


In [6]:

# Calculate explained variance
explained_variance = pca.explained_variance_ratio_

variance_table = pd.DataFrame({
    "Principal Component": [
        f"PC{i+1}" for i in range(len(explained_variance))
    ],
    "Explained Variance": explained_variance,
    "Cumulative Variance": np.cumsum(explained_variance)
})

display(variance_table)

print(
    "Total Variance Retained:",
    round(np.sum(explained_variance) * 100, 2),
    "%"
)


,Principal Component,Explained Variance,Cumulative Variance
0,PC1,0.276055,0.276055
1,PC2,0.193202,0.469256
2,PC3,0.133870,0.603126
3,PC4,0.080522,0.683648
4,PC5,0.064165,0.747813
5,PC6,0.045429,0.793242
6,PC7,0.039669,0.832911
7,PC8,0.029858,0.862769
8,PC9,0.024841,0.887609
9,PC10,0.021101,0.908710


Total Variance Retained: 95.34 %


In [7]:

# Initialize K-Means on PCA-reduced data
model = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

# Fit model on PCA training data
train_clusters = model.fit_predict(X_train_pca)

# Predict testing clusters
test_clusters = model.predict(X_test_pca)

print("PCA-Based Clustering Completed")

print("\nCluster Distribution:")
print(pd.Series(train_clusters).value_counts().sort_index())


PCA-Based Clustering Completed

Cluster Distribution:
0    1405
1    4119
2    3044
3    2897
Name: count, dtype: int64


In [8]:

# Map clusters to majority condition classes
cluster_mapping = {}

for cluster in range(4):
    labels = train.loc[
        train_clusters == cluster, TARGET
    ]

    if len(labels) > 0:
        cluster_mapping[cluster] = labels.mode().iloc[0]
    else:
        cluster_mapping[cluster] = train[TARGET].mode().iloc[0]

print("Cluster to Condition Mapping:")
print(cluster_mapping)


Cluster to Condition Mapping:
{0: 'Fair', 1: 'Fair', 2: 'Fair', 3: 'Fair'}


In [9]:

# Predict testing classes
y_pred = np.array([
    cluster_mapping[c] for c in test_clusters
])

y_test = test[TARGET].astype(str)

print("Predicted Classes:")
print(pd.Series(y_pred).value_counts())

print("\nActual Classes:")
print(y_test.value_counts())


Predicted Classes:
Fair    2644
Name: count, dtype: int64

Actual Classes:
Condition_Class
Fair         1788
Good          586
Poor          238
Excellent      32
Name: count, dtype: int64


In [10]:

# Evaluate PCA-based clustering
print("Principal Component Analysis with K-Means")

print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print("\nClassification Report:")

print(classification_report(
    y_test,
    y_pred,
    labels=CLASSES,
    zero_division=0
))

# Evaluate cluster separation
silhouette = silhouette_score(
    X_train_pca,
    train_clusters,
    sample_size=2000,
    random_state=42
)

print("Silhouette Score:", silhouette)

print(
    "Total Explained Variance:",
    pca.explained_variance_ratio_.sum()
)

print("Number of Principal Components:", pca.n_components_)


Principal Component Analysis with K-Means
Accuracy: 0.6762481089258698

Classification Report:
              precision    recall  f1-score   support

   Excellent       0.00      0.00      0.00        32
        Good       0.00      0.00      0.00       586
        Fair       0.68      1.00      0.81      1788
        Poor       0.00      0.00      0.00       238

    accuracy                           0.68      2644
   macro avg       0.17      0.25      0.20      2644
weighted avg       0.46      0.68      0.55      2644

Silhouette Score: 0.16684989212107673
Total Explained Variance: 0.9534443186321065
Number of Principal Components: 13


In [11]:
# Clustering is unsupervised: the accuracy above comes from mapping each cluster to its
# majority class, so compare it with the majority-class baseline and use label-free metrics.
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             silhouette_score, f1_score)
_y = test[TARGET].astype(str)
_X = X_train_pca if "X_train_pca" in globals() else X_train
print("Majority-class baseline accuracy:", round(_y.value_counts(normalize=True).max(), 4))
print("Macro F1 of cluster-to-class mapping:", round(f1_score(_y, y_pred, average="macro", zero_division=0), 4))
print("Adjusted Rand Index vs true labels (test):", round(adjusted_rand_score(_y, test_clusters), 4))
print("Normalized Mutual Information (test):", round(normalized_mutual_info_score(_y, test_clusters), 4))
print("Silhouette (train sample):", round(silhouette_score(_X, train_clusters, sample_size=2000, random_state=42), 4))


Majority-class baseline accuracy: 0.6762
Macro F1 of cluster-to-class mapping: 0.2017
Adjusted Rand Index vs true labels (test): 0.0723
Normalized Mutual Information (test): 0.0991


Silhouette (train sample): 0.1668


In [12]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

train_graph = pd.read_csv("../training_image_features_grouped.csv")

excluded = [
    "Condition_Class", "Condition_Score",
    "Damage_Score", "Alternative_Use_Suitability"
]

from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

X = train_graph[features].replace([np.inf, -np.inf], np.nan)
X = SimpleImputer(strategy="median").fit_transform(X)
X = StandardScaler().fit_transform(X)

pca_model = PCA()
pca_model.fit(X)

explained_variance = pca_model.explained_variance_ratio_

plt.figure(figsize=(10, 5))
plt.bar(
    range(1, len(explained_variance) + 1),
    explained_variance
)

plt.title("PCA: Individual Explained Variance")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_7176\3924664426.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

train_graph = pd.read_csv("../training_image_features_grouped.csv")

excluded = [
    "Condition_Class", "Condition_Score",
    "Damage_Score", "Alternative_Use_Suitability"
]

from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

X = train_graph[features].replace([np.inf, -np.inf], np.nan)
X = SimpleImputer(strategy="median").fit_transform(X)
X = StandardScaler().fit_transform(X)

pca_model = PCA()
pca_model.fit(X)

cumulative_variance = np.cumsum(
    pca_model.explained_variance_ratio_
)

plt.figure(figsize=(10, 5))
plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance,
    marker="o"
)

plt.axhline(
    y=0.95,
    linestyle="--",
    label="95% Variance"
)

plt.title("PCA: Cumulative Explained Variance")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

components_95 = np.argmax(cumulative_variance >= 0.95) + 1
print("Components required for 95% variance:", components_95)


Components required for 95% variance: 13


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_7176\4072867028.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

train_graph = pd.read_csv("../training_image_features_grouped.csv")

excluded = [
    "Condition_Class", "Condition_Score",
    "Damage_Score", "Alternative_Use_Suitability"
]

from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

X = train_graph[features].replace([np.inf, -np.inf], np.nan)
X = SimpleImputer(strategy="median").fit_transform(X)
X = StandardScaler().fit_transform(X)

# Use PCA fitted on training data
X_2d = pca.transform(X)

# Use first two principal components
X_2d = X_2d[:, :2]

classes = ["Excellent", "Good", "Fair", "Poor"]
y = train_graph["Condition_Class"].reset_index(drop=True)

plt.figure(figsize=(10, 7))

for condition in classes:
    mask = (y == condition).to_numpy()

    plt.scatter(
        X_2d[mask, 0],
        X_2d[mask, 1],
        label=condition,
        s=12,
        alpha=0.6
    )

plt.title("PCA: 2D Visualization of Material Conditions")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Condition")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(
    "Explained variance:",
    round(sum(pca_model.explained_variance_ratio_), 4)
)


Explained variance: 1.0


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_7176\2636461764.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

train_graph = pd.read_csv("../training_image_features_grouped.csv")

excluded = [
    "Condition_Class", "Condition_Score",
    "Damage_Score", "Alternative_Use_Suitability"
]

from pbl2_common import ORIGINAL_FEATURES
features = list(ORIGINAL_FEATURES)

X = train_graph[features].replace([np.inf, -np.inf], np.nan)
X = SimpleImputer(strategy="median").fit_transform(X)
X = StandardScaler().fit_transform(X)

pca_model = PCA()
pca_model.fit(X)

number_of_components = min(10, pca_model.components_.shape[0])
number_of_features = min(20, len(features))

plt.figure(figsize=(14, 7))

plt.imshow(
    pca_model.components_[
        :number_of_components, :number_of_features
    ],
    aspect="auto",
    cmap="coolwarm"
)

plt.colorbar(label="Component Loading")

plt.xticks(
    range(number_of_features),
    features[:number_of_features],
    rotation=90
)

plt.yticks(
    range(number_of_components),
    [f"PC{i+1}" for i in range(number_of_components)]
)

plt.title("PCA: Principal Component Loading Heatmap")
plt.xlabel("Original Features")
plt.ylabel("Principal Components")
plt.tight_layout()
plt.show()


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_7176\3158805873.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
